# Lezione 2. Visione artificiale I: reti convoluzionali

**Advanced Deep Learning**

*Sigle usate in questa lezione: rete neurale convoluzionale (CNN), unità lineare rettificata (ReLU), unità di elaborazione grafica (GPU), rosso-verde-blu (RGB), discesa del gradiente stocastica (SGD).*

## Obiettivi della lezione

Al termine di questa lezione saprai:

- Spiegare perché una rete completamente connessa non funziona bene con le immagini
- Descrivere come agisce un filtro convoluzionale su un'immagine, passo dopo passo
- Calcolare la dimensione di output di un livello convoluzionale a partire dal *kernel*, dallo *stride* e dal *padding*
- Spiegare a cosa serve il *pooling* all'interno di una CNN
- Costruire e addestrare una piccola rete neurale convoluzionale (CNN) con PyTorch su FashionMNIST
- Visualizzare che cosa ha imparato il primo livello di una CNN già addestrata

## Perché una rete normale non basta per le immagini

- Per dare un'immagine a una rete completamente connessa, prima la appiattisci (*flatten*): trasformi la griglia di pixel in un unico vettore
- Appiattendo, perdi la struttura spaziale: due pixel vicini nell'immagine originale possono finire molto distanti nel vettore
- Il numero di parametri esplode. Un'immagine a colori (RGB) di 32 x 32 pixel ha 3.072 valori di input; collegati a 1.000 neuroni del primo livello nascosto fanno già 3.072.000 parametri
- Le immagini naturali hanno due proprietà che una CNN sfrutta:
  - **Località spaziale**: i pixel vicini sono spesso correlati (appartengono allo stesso oggetto o alla stessa texture)
  - **Invarianza alla traslazione**: un pattern (un occhio, un bordo) può comparire in qualunque posizione dell'immagine
- Un livello convoluzionale sfrutta entrambe le proprietà:
  - Collega ogni unità di output solo con una regione locale dell'input, detta **campo recettivo**
  - Riutilizza gli stessi pesi su tutta l'immagine: **condivisione dei pesi** (*weight sharing*)

## Il filtro convoluzionale in azione

- Un filtro (*kernel*) è una piccola matrice di pesi, per esempio di 3 x 3
- Scorre sull'immagine: in ogni posizione moltiplica i propri valori per i pixel sottostanti e somma il risultato
- Se la regione somiglia al pattern del filtro, la somma dà un valore alto; se non vi somiglia, dà un valore basso
- Il risultato di percorrere tutta l'immagine è una **mappa di attivazioni** (*feature map*)
- In una CNN addestrata, i valori del filtro non li progetta una persona: li impara la rete, esattamente come i pesi di un livello lineare

Qui sotto applichi a mano, con NumPy, un rilevatore di bordi verticali su una piccola immagine sintetica. Fai attenzione alla dimensione di output: con un input di 9 x 9 e un filtro di 3 x 3 senza padding, ottieni 7 x 7 (convoluzione "valida").

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Imagen sintética de 9x9: mitad izquierda oscura, mitad derecha clara -> un borde vertical
imagen = np.zeros((9, 9), dtype=np.float32)
imagen[:, 5:] = 1.0

# Filtro detector de bordes verticales (tipo Sobel)
kernel = np.array([
    [-1, 0, 1],
    [-2, 0, 2],
    [-1, 0, 1],
], dtype=np.float32)

def convolucion_manual(imagen, kernel):
    f = kernel.shape[0]
    n = imagen.shape[0]
    salida = np.zeros((n - f + 1, n - f + 1), dtype=np.float32)
    for i in range(salida.shape[0]):
        for j in range(salida.shape[1]):
            region = imagen[i:i + f, j:j + f]
            salida[i, j] = np.sum(region * kernel)
    return salida

mapa_activaciones = convolucion_manual(imagen, kernel)

fig, ejes = plt.subplots(1, 2, figsize=(8, 4))
ejes[0].imshow(imagen, cmap="gray")
ejes[0].set_title("Imagen original (borde vertical)")
ejes[1].imshow(mapa_activaciones, cmap="gray")
ejes[1].set_title("Mapa de activaciones")
for eje in ejes:
    eje.axis("off")
plt.tight_layout()
plt.show()

print(f"Tamaño de entrada: {imagen.shape} | Tamaño de salida: {mapa_activaciones.shape}")

## Stride, padding e pooling

- **Convoluzione valida (*valid*)**: senza padding, l'output è sempre più piccolo dell'input: $O = N - F + 1$
- **Padding (riempimento)**: aggiungi zeri attorno al bordo dell'immagine; con il padding giusto ottieni una convoluzione "uguale" (*same*), che mantiene la stessa dimensione in input e in output
- **Stride (passo)**: controlla ogni quanti pixel il filtro si sposta; con *stride* due calcoli solo la metà delle posizioni in ciascuna dimensione
- Formula generale, combinando *kernel* ($F$), *padding* ($P$) e *stride* ($S$): $O = \lfloor (N - F + 2P) / S \rfloor + 1$
- **Pooling** (tipicamente *max pooling* 2 x 2) riduce la risoluzione spaziale delle mappe di attivazione senza aggiungere parametri da apprendere

In [ ]:
import torch
import torch.nn as nn

entrada = torch.randn(1, 1, 9, 9)  # (lote, canales, alto, ancho)
print(f"Tamaño de entrada: {tuple(entrada.shape)}\n")

configuraciones = [
    {"kernel_size": 3, "stride": 1, "padding": 0},  # convolución válida
    {"kernel_size": 3, "stride": 1, "padding": 1},  # convolución "igual" (same)
    {"kernel_size": 3, "stride": 2, "padding": 0},  # stride dos
    {"kernel_size": 5, "stride": 1, "padding": 2},  # kernel más grande, "igual"
]

for config in configuraciones:
    capa = nn.Conv2d(in_channels=1, out_channels=1, **config)
    salida = capa(entrada)
    print(f"kernel={config['kernel_size']} stride={config['stride']} "
          f"padding={config['padding']} -> salida {tuple(salida.shape)}")

# Pooling: reduce alto y ancho a la mitad, sin parámetros que aprender
pool = nn.MaxPool2d(kernel_size=2, stride=2)
mapa_ejemplo = torch.randn(1, 32, 32, 32)
print(f"\nAntes de max pooling: {tuple(mapa_ejemplo.shape)}")
print(f"Después de max pooling 2x2: {tuple(pool(mapa_ejemplo).shape)}")

## Costruisci la tua CNN

- Una CNN sovrappone blocchi di convoluzione + ReLU + *pooling* e termina con uno o più livelli completamente connessi
- Addestri il modello su FashionMNIST: immagini in scala di grigi di 28 x 28 pixel con dieci categorie di capi di abbigliamento, disponibile direttamente in `torchvision`
- L'addestramento segue la stessa procedura che già conosci: perdita (entropia incrociata), retropropagazione e aggiornamento dei pesi con un ottimizzatore (qui Adam, una variante di SGD)
- Se Colab ti assegna una GPU, l'addestramento è più rapido; altrimenti funziona ugualmente su CPU

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Dispositivo de entrenamiento: {device}")

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.2860,), (0.3530,)),
])

train_dataset = datasets.FashionMNIST(
    root="./data", train=True, download=True, transform=transform
)
test_dataset = datasets.FashionMNIST(
    root="./data", train=False, download=True, transform=transform
)

train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=256, shuffle=False)

clases = [
    "camiseta", "pantalón", "jersey", "vestido", "abrigo",
    "sandalia", "camisa", "zapatilla", "bolso", "botín",
]

In [ ]:
class CNNSimple(nn.Module):
    def __init__(self, num_clases=10):
        super().__init__()
        # Bloque 1: 1 canal de entrada (escala de grises) -> 16 filtros, padding "same"
        self.conv1 = nn.Conv2d(in_channels=1, out_channels=16,
                                kernel_size=3, stride=1, padding=1)
        # Bloque 2: 16 -> 32 filtros
        self.conv2 = nn.Conv2d(in_channels=16, out_channels=32,
                                kernel_size=3, stride=1, padding=1)
        # Max pooling 2x2: reduce el alto y el ancho a la mitad cada vez
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)

        # Dos pooling de 2x2 sobre una entrada de 28x28: 28 -> 14 -> 7
        self.fc1 = nn.Linear(32 * 7 * 7, 128)
        self.fc2 = nn.Linear(128, num_clases)
        self.dropout = nn.Dropout(0.25)  # mitiga el sobreajuste

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))  # -> (lote, 16, 14, 14)
        x = self.pool(F.relu(self.conv2(x)))  # -> (lote, 32, 7, 7)
        x = torch.flatten(x, start_dim=1)     # aplana todo menos el eje de lote
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)                       # logits, sin softmax explícito
        return x

modelo = CNNSimple(num_clases=10).to(device)
print(modelo)

In [ ]:
criterio = nn.CrossEntropyLoss()
optimizador = torch.optim.Adam(modelo.parameters(), lr=1e-3)

def entrenar_una_epoca(modelo, loader, criterio, optimizador):
    modelo.train()
    perdida_total = 0.0
    for imagenes, etiquetas in loader:
        imagenes, etiquetas = imagenes.to(device), etiquetas.to(device)
        optimizador.zero_grad()
        salidas = modelo(imagenes)
        perdida = criterio(salidas, etiquetas)
        perdida.backward()
        optimizador.step()
        perdida_total += perdida.item() * imagenes.size(0)
    return perdida_total / len(loader.dataset)

def evaluar(modelo, loader, criterio):
    modelo.eval()
    perdida_total = 0.0
    aciertos = 0
    with torch.no_grad():
        for imagenes, etiquetas in loader:
            imagenes, etiquetas = imagenes.to(device), etiquetas.to(device)
            salidas = modelo(imagenes)
            perdida_total += criterio(salidas, etiquetas).item() * imagenes.size(0)
            predicciones = salidas.argmax(dim=1)
            aciertos += (predicciones == etiquetas).sum().item()
    return perdida_total / len(loader.dataset), aciertos / len(loader.dataset)

# Pocas épocas: en clase priorizamos ver el proceso completo funcionando, no exprimir la exactitud
num_epocas = 3
perdidas_train = []
perdidas_test = []

for epoca in range(1, num_epocas + 1):
    perdida_train = entrenar_una_epoca(modelo, train_loader, criterio, optimizador)
    perdida_test, exactitud_test = evaluar(modelo, test_loader, criterio)
    perdidas_train.append(perdida_train)
    perdidas_test.append(perdida_test)
    print(f"Época {epoca}/{num_epocas} | "
          f"pérdida train: {perdida_train:.4f} | "
          f"pérdida test: {perdida_test:.4f} | "
          f"exactitud test: {exactitud_test:.2%}")

print(f"\nExactitud final sobre el conjunto de prueba: {exactitud_test:.2%}")

In [ ]:
plt.figure(figsize=(6, 4))
plt.plot(range(1, num_epocas + 1), perdidas_train, marker="o", label="Pérdida train")
plt.plot(range(1, num_epocas + 1), perdidas_test, marker="o", label="Pérdida test")
plt.xlabel("Época")
plt.ylabel("Pérdida (entropía cruzada)")
plt.title("Curva de pérdida durante el entrenamiento")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## Che cosa ha imparato la rete

- Il primo livello convoluzionale impara quasi sempre rilevatori semplici: bordi e variazioni di intensità
- Puoi visualizzare direttamente i pesi di questi filtri, perché ciascuno è una piccola matrice, come il `kernel` che hai applicato a mano all'inizio della lezione

In [ ]:
pesos_conv1 = modelo.conv1.weight.data.cpu().clone()  # forma (16, 1, 3, 3)
num_filtros = pesos_conv1.shape[0]

fig, ejes = plt.subplots(2, 8, figsize=(12, 3.5))
for indice, eje in enumerate(ejes.flat):
    filtro = pesos_conv1[indice, 0]
    eje.imshow(filtro, cmap="gray")
    eje.set_title(f"Filtro {indice}", fontsize=8)
    eje.axis("off")

plt.suptitle("Filtros aprendidos en la primera capa convolucional")
plt.tight_layout()
plt.show()

## Riepilogo della lezione

- Una rete completamente connessa non scala alle immagini: distrugge la struttura spaziale quando appiattisce l'input e richiede troppi parametri
- Un filtro convoluzionale è un modello di pesi apprendibili che scorre sull'immagine alla ricerca di un pattern
- La condivisione dei pesi riduce drasticamente il numero di parametri e rende la rete equivariante rispetto alla traslazione
- Lo *stride*, il *padding* e il *pooling* controllano la dimensione spaziale delle mappe di attivazione lungo la rete
- Una CNN si costruisce sovrapponendo blocchi di convoluzione, ReLU e *pooling*, e termina con livelli completamente connessi
- Man mano che si sovrappongono livelli convoluzionali, il campo recettivo cresce: prima bordi e texture, poi forme, infine oggetti interi

**Apri ora il quaderno di pratica `../notebooks_practica/clase02_practica_cnn_vision_por_computador.ipynb` e mettilo in funzione.**